In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import os
import warnings
warnings.filterwarnings('ignore')

PARQUET_PATH = 'signals.parquet'
OUTPUT_DIR   = 'prepared_data/v2_1024_FINAL22'
os.makedirs(OUTPUT_DIR, exist_ok=True)

SAMPLING_RATE   = 51200
SEGMENT_SAMPLES = 1024

COL_FEATURE  = 'Feature'
COL_BANDYMAS = 'Bandymas'
COL_APKROVA  = 'Apkrova(Nm)'
COL_SUKIAI   = 'Sukiai(rpm)'
COL_SIGNAL   = 'Value'

VAL_SIZE     = 0.2
RANDOM_STATE = 42

print(f'Segmento ilgis : {SEGMENT_SAMPLES} reiksmiu ({1000*SEGMENT_SAMPLES/SAMPLING_RATE:.1f} ms)')

## 1. Failo uzkrovimas ir signalu rekonstravimas

In [ ]:
print('Uzkraunamas parquet...')
df = pd.read_parquet(PARQUET_PATH)
df[COL_FEATURE]  = df[COL_FEATURE].str.replace('Feat', '').astype(int)
df[COL_BANDYMAS] = df[COL_BANDYMAS].str.extract(r'(\d+)').astype(int)
print(f'Uzkrauta: {df.shape[0]:,} eiluciu')

print('Rekonstruojami signalai...')
signal_groups = df.groupby(
    [COL_FEATURE, COL_BANDYMAS, COL_APKROVA, COL_SUKIAI]
)[COL_SIGNAL].apply(np.array).reset_index()
signal_groups.columns = ['label', 'batch', 'load', 'rpm', 'signal']

print(f'Is viso signalu : {len(signal_groups)}')
print(f'Signalo ilgis : {len(signal_groups.iloc[0]["signal"]):,} reiksmes')
print(f'\nSignalu per klase:')
print(signal_groups.groupby('label').size().to_string())
print(f'\nSignalu per bandyma:')
print(signal_groups.groupby('batch').size().to_string())

In [ ]:
def build_references(signal_groups):
    references = {}
    b1 = signal_groups[signal_groups['batch'] == 1]
    for (label, load, rpm), group in b1.groupby(['label', 'load', 'rpm']):
        stacked = np.stack(group['signal'].values).astype(np.float32)
        references[(label, load, rpm)] = stacked.mean(axis=0)
    return references

references = build_references(signal_groups)
print(f'Sudaryta {len(references)} signalu kuriais bus remiamasi (po viena per gedimo/apkrovos/sukiu kombinacija)')

In [ ]:
def find_alignment_lag(signal, reference, max_lag):
    search_len = max_lag * 4
    sig_chunk  = signal[:search_len].astype(np.float64)
    ref_chunk  = reference[:search_len].astype(np.float64)

    corr = np.correlate(sig_chunk, ref_chunk, mode='full')
    center = len(ref_chunk) - 1
    corr_window = corr[center : center + max_lag + 1]
    lag = int(np.argmax(corr_window))
    return lag


def align_batch2_signals(signal_groups, references, sampling_rate):
    aligned_rows = []

    for _, row in signal_groups.iterrows():
        sig   = row['signal'].astype(np.float32)
        batch = int(row['batch'])
        rpm   = row['rpm']
        key   = (row['label'], row['load'], rpm)

        if batch == 2 and key in references:
            samples_per_rotation = int(sampling_rate / (rpm / 60))
            lag = find_alignment_lag(sig, references[key], samples_per_rotation)
            sig = sig[lag:]
            print(f'  {key} → atsilikimas={lag} reiksmiu ({lag/sampling_rate*1000:.2f} ms), '
                  f'rotacija per suki={samples_per_rotation} reiksmiu')
        else:
            lag = 0

        aligned_rows.append({
            'label': row['label'],
            'batch': row['batch'],
            'load':  row['load'],
            'rpm':   rpm,
            'signal': sig,
            'lag':   lag
        })

    min_len_b2 = min(len(r['signal']) for r in aligned_rows if r['batch'] == 2)
    for r in aligned_rows:
        if r['batch'] == 2:
            r['signal'] = r['signal'][:min_len_b2]

    original_len = len(signal_groups.iloc[0]['signal'])
    print(f'\nTrumpiausias signalas po sinchronizavimo: {min_len_b2:,} reiksmes')
    print(f'Reiksmiu prarasta nuo pradinio signalo {original_len:,}: {original_len - min_len_b2}')

    return pd.DataFrame(aligned_rows)


print('Sinchronizuojami 2 bandymo signalai...')
signal_groups = align_batch2_signals(signal_groups, references, SAMPLING_RATE)

b2_lags = signal_groups[signal_groups['batch'] == 2]['lag']
print(f'\n2 bandymo signalu atsilikimai:')
print(f'  min={b2_lags.min()}, max={b2_lags.max()}, vid={b2_lags.mean():.1f} reiksmes')

## 4. Segmentavimas sinchronizuotu signalu

In [ ]:
def segment_signal(sig, segment_samples):
    n_segs = len(sig) // segment_samples
    return [sig[i*segment_samples : (i+1)*segment_samples] for i in range(n_segs)]

def segment_all_signals(signal_groups, segment_samples):
    segments, labels, batches, rpms, loads = [], [], [], [], []

    for _, row in signal_groups.iterrows():
        sig   = row['signal']
        label = int(row['label'])
        batch = int(row['batch'])
        rpm   = row['rpm']
        load  = row['load']

        windows = segment_signal(sig, segment_samples)
        for w in windows:
            segments.append(w)
            labels.append(label)
            batches.append(batch)
            rpms.append(rpm)
            loads.append(load)

    return (
        np.stack(segments),
        np.array(labels,  dtype=np.int64),
        np.array(batches, dtype=np.int64),
        np.array(rpms,    dtype=np.int64),
        np.array(loads,   dtype=np.int64)
    )


print('Segmentavimas...')
X_all, y_all, batch_all, rpm_all, load_all = segment_all_signals(signal_groups, SEGMENT_SAMPLES)

print(f'\nIs viso segmentu : {len(X_all):,}')
print(f'Segmentu forma  : {X_all.shape}')
print(f'\nSegmentu pasiskirstymas pagal gedima:')
for c, n in zip(*np.unique(y_all, return_counts=True)):
    print(f'  Gedimas {c}: {n:,}  ({100*n/len(y_all):.1f}%)')

## 5. Treniravimo/validavimo/testavimo aibiu sudarymas

In [ ]:
test_mask  = batch_all == 2
train_mask = batch_all == 1

X_test, y_test = X_all[test_mask], y_all[test_mask]
rpm_test, load_test = rpm_all[test_mask], load_all[test_mask]

X_b1, y_b1 = X_all[train_mask], y_all[train_mask]
rpm_b1, load_b1 = rpm_all[train_mask], load_all[train_mask]

np.save(f'{OUTPUT_DIR}/X_batch1.npy', X_b1)
np.save(f'{OUTPUT_DIR}/y_batch1.npy', y_b1)
np.save(f'{OUTPUT_DIR}/rpm_batch1.npy', rpm_b1)
np.save(f'{OUTPUT_DIR}/load_batch1.npy', load_b1)
print(f'X_batch1 : {X_b1.shape}  ← Pilna 1 bandymo aibe nepakeista segmentu tvarka')

train_idx, val_idx = [], []
for cls in np.unique(y_b1):
    cls_indices = np.where(y_b1 == cls)[0]
    split_point = int(len(cls_indices) * (1 - VAL_SIZE))
    train_idx.extend(cls_indices[:split_point])
    val_idx.extend(cls_indices[split_point:])

train_idx = np.array(train_idx)
val_idx   = np.array(val_idx)

X_train, y_train = X_b1[train_idx], y_b1[train_idx]
X_val,   y_val   = X_b1[val_idx],   y_b1[val_idx]

print(f'X_train : {X_train.shape}  ← pirmi 80% segmentu')
print(f'X_val   : {X_val.shape}  ← paskutiniai 20% segmentu')
print(f'X_test  : {X_test.shape}  ← 2 bandymas sulygintas su pirmuoju')

print(f'\nMokymo aibes segmentu pasiskirstymas pagal gedimus:')
for c, n in zip(*np.unique(y_train, return_counts=True)):
    print(f'  Gedimas {c}: {n:,}  ({100*n/len(y_train):.1f}%)')
print(f'\nValidavimo aibes segmentu pasiskirstymas pagal gedimus:')
for c, n in zip(*np.unique(y_val, return_counts=True)):
    print(f'  Gedimas {c}: {n:,}  ({100*n/len(y_val):.1f}%)')

## 7. Issaugojimas

In [ ]:
print('Saving...')

np.save(f'{OUTPUT_DIR}/X_train.npy', X_train)
np.save(f'{OUTPUT_DIR}/X_val.npy',   X_val)
np.save(f'{OUTPUT_DIR}/X_test.npy',  X_test)
np.save(f'{OUTPUT_DIR}/y_train.npy', y_train)
np.save(f'{OUTPUT_DIR}/y_val.npy',   y_val)
np.save(f'{OUTPUT_DIR}/y_test.npy',  y_test)
np.save(f'{OUTPUT_DIR}/rpm_test.npy',  rpm_test)
np.save(f'{OUTPUT_DIR}/load_test.npy', load_test)

print(f'\nVisi failai issaugoti: {OUTPUT_DIR}/')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    if fname.endswith('.npy'):
        size_mb = os.path.getsize(f'{OUTPUT_DIR}/{fname}') / 1e6
        print(f'  {fname:<25} {size_mb:.1f} MB')